In [20]:
# imports
import pandas as pd
import numpy as np
import requests
import time
from bs4 import BeautifulSoup
import re
from selenium import webdriver

In [21]:
# Új dataframe létrehozása
if input("Ha tényleg új dataframe-t szeretnél, írd le hogy 'Igen'") == "Igen":
    cities = {
        "Barcelona": {"country": "Spain"},
        "Lisbon": {"country": "Portugal"},
        "Tirana": {"country": "Albania"},
        "Geneva": {"country": "Switzerland"},
        "Antalya": {"country": "Turkey"},
        "Amsterdam": {"country": "Netherlands"},
        "Berlin": {"country": "Germany"},
        "Paris": {"country": "France"},
        "Rome": {"country": "Italy"},
        "Madrid": {"country": "Spain"},
        "Athens": {"country": "Greece"},
        "Vienna": {"country": "Austria"},
        "London": {"country": "United Kingdom"},
        "Warsaw": {"country": "Poland"},
        "Brussels": {"country": "Belgium"},
        "Prague": {"country": "Czech Republic"},
        "Milan": {"country": "Italy"},
        "Copenhagen": {"country": "Denmark"},
        "Dubrovnik": {"country": "Croatia"},
        "Oslo": {"country": "Norway"}
    }

    df = pd.DataFrame(cities).T
    print("Új dataframe:")
    display(df.sample(5))
else:
    print("Új dataframe készítése megszakítva.")

Új dataframe:


,country
London,United Kingdom
Tirana,Albania
Barcelona,Spain
Madrid,Spain
Athens,Greece


In [22]:
# Földrajz API

# Fő földrajzi típusok Overpass kulcsszavai
geo_types = {
    "beach": "natural=beach",
    "mountain": "natural=peak",
    "lake": "natural=lake",
    "desert": "natural=desert",
    "island": "place=island",
    "attraction": "tourism=attraction",
    "park": "leisure=park", 
    "monument": "historic=monument",
}

def get_coordinates(city, country):
    """
    Visszaadja a (lat, lon) koordinátákat egy város és ország alapján
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "city": city,
        "country": country,
        "format": "json",
        "limit": 1
    }
    
    response = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    
    if response.status_code == 200 and response.json():
        data = response.json()[0]
        lat = float(data["lat"])
        lon = float(data["lon"])
        return lat, lon
    else:
        return None, None
    
def get_geo_scores(lat, lon, radius=10000):
    """
    Lekérdezi az Overpass API-t, és visszaadja a fő földrajzi típusokra
    a találatok számát normalizált 0-1 skálán.
    """
    scores = {}
    
    for typ, tag in geo_types.items():
        query = f"""
        [out:json][timeout:25];
        (
          node["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          way["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          relation["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
        );
        out center;
        """
        url = "http://overpass-api.de/api/interpreter"
        response = requests.post(url, data={"data": query})

        time.sleep(0.5)  # rate limit elkerülése
        
        if response.status_code == 200:
            data = response.json()
            count = len(data["elements"])
            scores[typ] = count
        else:
            scores[typ] = -1  # hiba esetén -1
    
    return scores

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row["country"])
    scores = get_geo_scores(lat, lon)
    print(f"\n{city} földrajzi típus pontszámok:")
    for k, v in scores.items():
        df.loc[city, f"geo_{k}"] = v
        print(f"  {k}: {v:.2f}")



Barcelona földrajzi típus pontszámok:
  beach: 23.00
  mountain: 58.00
  lake: 0.00
  desert: 0.00
  island: 1.00
  attraction: 126.00
  park: 1003.00
  monument: 115.00

Lisbon földrajzi típus pontszámok:
  beach: 30.00
  mountain: 6.00
  lake: 0.00
  desert: 0.00
  island: 0.00
  attraction: 247.00
  park: 669.00
  monument: 27.00

Tirana földrajzi típus pontszámok:
  beach: 0.00
  mountain: 160.00
  lake: 0.00
  desert: 0.00
  island: 0.00
  attraction: 47.00
  park: 255.00
  monument: 16.00

Geneva földrajzi típus pontszámok:
  beach: 42.00
  mountain: 3.00
  lake: 0.00
  desert: 0.00
  island: 0.00
  attraction: 73.00
  park: 1002.00
  monument: 11.00

Antalya földrajzi típus pontszámok:
  beach: 14.00
  mountain: 2.00
  lake: 0.00
  desert: 0.00
  island: 0.00
  attraction: 21.00
  park: 752.00
  monument: 15.00

Amsterdam földrajzi típus pontszámok:
  beach: 31.00
  mountain: 8.00
  lake: 0.00
  desert: 0.00
  island: 6.00
  attraction: 152.00
  park: 364.00
  monument: 1.00

B

In [23]:
# Költségek (numbeo)

def clean_col_name(name):
    """
    Tisztítja az oszlopneveket:
    - kisbetűs
    - minden speciális karaktert aláhúzásra cserél
    - többszörös aláhúzásból egyet csinál
    """
    name = name.lower()
    # Cseréljük a nem alfanumerikus karaktereket aláhúzásra
    name = re.sub(r'[^a-z0-9]+', '_', name)
    # Többszörös aláhúzás → 1 aláhúzás
    name = re.sub(r'_+', '_', name)
    # Elejéről és végéről aláhúzás eltávolítása
    name = name.strip('_')
    return name


for city, row in df.iterrows():
    # URL encode a városnévhez, ha van szóköz
    city_url = city.replace(" ", "-")
    url = f"https://www.numbeo.com/cost-of-living/in/{city_url}?displayCurrency=EUR"

    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", {"class": "data_wide_table"})
    if table is None:
        print("  Nincs adat a városhoz.")
        continue

    rows = table.find_all("tr")
    for tr in rows:
        cols = tr.find_all("td")
        if len(cols) >= 2:
            item = cols[0].text.strip()
            value = cols[1].text.strip()
            try:
                value_num = float(value.replace("€","").replace(",","").strip())
            except:
                value_num = None

            # Tisztított oszlopnév
            col_name = "col_" + clean_col_name(item)
            df.loc[city, col_name] = value_num


In [24]:
# Klíma

def get_hourly_weather(latitude, longitude, start_date, end_date):
    # pip install openmeteo-requests
    # pip install requests_cache
    # pip install retry-requests
    
    import openmeteo_requests
    import requests_cache
    import pandas as pd
    from retry_requests import retry
    
    # Open-Meteo API kliens beállítása gyorsítótárral és hibakezeléssel
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # API hívás paramétereinek beállítása
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,  # szélességi fok
        "longitude": longitude,  # hosszúsági fok
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,wind_speed_10m,weathercode"
    }
    
    # API hívás végrehajtása
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    
    # Óránkénti adatok feldolgozása
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(1).ValuesAsNumpy()
    hourly_weathercode = hourly.Variables(2).ValuesAsNumpy()
    
    # Időbélyegek létrehozása
    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=False),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=False),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature_2m": hourly_temperature_2m,
        "wind_speed_10m": hourly_wind_speed_10m,
        "weathercode": hourly_weathercode
    }
    
    # DataFrame létrehozása
    hourly_dataframe = pd.DataFrame(data=hourly_data)
    
    # Időjárás kódok értelmezése
    weather_descriptions = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Intense drizzle",
        56: "Light freezing drizzle",
        57: "Intense freezing drizzle",
        61: "Light rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Light snow",
        73: "Moderate snow",
        75: "Heavy snow",
        77: "Hail",
        80: "Light showers",
        81: "Moderate showers",
        82: "Heavy showers",
        85: "Light snow showers",
        86: "Heavy snow showers",
        95: "Light or moderate thunderstorm",
        96: "Light thunderstorm with hail",
        99: "Severe thunderstorm with hail"
    }
    
    # Az időjárás kódok leírásának hozzáadása a DataFrame-hez
    hourly_dataframe["weather_description"] = hourly_dataframe["weathercode"].map(weather_descriptions)
    # Átváltás m/s-ról km/h-ra
    hourly_dataframe["wind_speed_kmh"] = hourly_dataframe["wind_speed_10m"] * 3.6
    
    # Eredmény kiíratása
    output_df = hourly_dataframe[["date", "temperature_2m", "wind_speed_kmh", "weather_description", "weathercode"]]
    output_df = output_df.copy()
    output_df.rename(columns={"temperature_2m": "temp_celsius"}, inplace=True)
        
    return output_df

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row['country'])
    if lat is None:
        continue

    # archive API (példa 2023-as év)
    climate_df = get_hourly_weather(lat, lon, "2023-01-01", "2023-12-31")

    # Egyszerű havi aggregáció (átlag hőmérséklet)
    climate_df['month'] = climate_df['date'].dt.month
    monthly_avg = climate_df.groupby('month')['temp_celsius'].mean()

    for month, value in monthly_avg.items():
        col_name = f"climate_temp_mean_{month}"
        df.loc[city, col_name] = value

print(df[[col for col in df.columns if 'climate_temp' in col][:3]].head(5))


           climate_temp_mean_1  climate_temp_mean_2  climate_temp_mean_3
Barcelona             8.339535             8.767385            13.490542
Lisbon               11.784344            11.326229            14.246442
Tirana                8.549989             7.234667            11.243941
Geneva                3.000288             4.255777             7.547465
Antalya              11.532994             9.479650            13.863034


In [25]:
# Életstílus

# Dimenziók és OSM kulcs-érték párok
lifestyle_categories = {
    "bulis": [
        {"key": "amenity", "value": "bar"},
        {"key": "amenity", "value": "pub"},
        {"key": "amenity", "value": "nightclub"},
    ],
    "relax": [
        {"key": "leisure", "value": "park"},
        {"key": "natural", "value": "beach"},
        {"key": "leisure", "value": "garden"},
    ],
    "kulturalis": [
        {"key": "tourism", "value": "museum"},
        {"key": "tourism", "value": "artwork"},
        {"key": "historic", "value": "monument"},
    ],
    "csaladbarat": [
        {"key": "amenity", "value": "kindergarten"},
        {"key": "leisure", "value": "playground"},
        {"key": "tourism", "value": "zoo"},
    ]
}

import time

def query_overpass(lat, lon, key, value, radius=10000):
    overpass_url = "http://overpass-api.de/api/interpreter"
    query = f"""
    [out:json][timeout:25];
    (
      node["{key}"="{value}"](around:{radius},{lat},{lon});
      way["{key}"="{value}"](around:{radius},{lat},{lon});
      relation["{key}"="{value}"](around:{radius},{lat},{lon});
    );
    out center;
    """
    response = requests.post(overpass_url, data={"data": query})
    if response.status_code == 200:
        data = response.json()
        return len(data.get("elements", []))
    else:
        return 0

def get_lifestyle_scores(city, country, radius=10000):
    lat, lon = get_coordinates(city, country)
    if not lat or not lon:
        return None

    scores = {}
    for dimension, cat_list in lifestyle_categories.items():
        count = 0
        for cat in cat_list:
            count += query_overpass(lat, lon, cat["key"], cat["value"], radius)
            time.sleep(1)  # rate limit miatt
        # egyszerű normalizálás: max 0-1 (tetszőleges normalizációs logika később)
        scores[dimension] = count
    return scores

for city, row in df.iterrows():
    scores = get_lifestyle_scores(city, row['country'], radius=10000)
    print(f"\n{city} életstílus pontszámok:")
    for dim, val in scores.items():
        df.loc[city, f"eletstilus_{dim}"] = val
        print(f"  {dim}: {val}")



Barcelona életstílus pontszámok:
  bulis: 1817
  relax: 4387
  kulturalis: 1127
  csaladbarat: 1589

Lisbon életstílus pontszámok:
  bulis: 507
  relax: 2466
  kulturalis: 1002
  csaladbarat: 886

Tirana életstílus pontszámok:
  bulis: 424
  relax: 438
  kulturalis: 80
  csaladbarat: 218

Geneva életstílus pontszámok:
  bulis: 236
  relax: 12443
  kulturalis: 515
  csaladbarat: 852

Antalya életstílus pontszámok:
  bulis: 59
  relax: 772
  kulturalis: 74
  csaladbarat: 164

Amsterdam életstílus pontszámok:
  bulis: 675
  relax: 10002
  kulturalis: 1117
  csaladbarat: 1566

Berlin életstílus pontszámok:
  bulis: 1785
  relax: 4254
  kulturalis: 2234
  csaladbarat: 4765

Paris életstílus pontszámok:
  bulis: 2772
  relax: 5300
  kulturalis: 3082
  csaladbarat: 2836

Rome életstílus pontszámok:
  bulis: 585
  relax: 2826
  kulturalis: 903
  csaladbarat: 438

Madrid életstílus pontszámok:
  bulis: 2535
  relax: 2110
  kulturalis: 676
  csaladbarat: 2442

Athens életstílus pontszámok:
  bu

In [26]:
# Távolság
 
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

start_city, start_country = "Budapest", "Hungary"
lat_start, lon_start = get_coordinates(start_city, start_country)

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row['country'])
    dist = haversine(lat_start, lon_start, lat, lon)

    df.loc[city, 'distance'] = dist
    print(f"{start_city}-{city} távolság:", round(dist, 1), "km")


Budapest-Barcelona távolság: 1497.3 km
Budapest-Lisbon távolság: 2470.1 km
Budapest-Tirana távolság: 688.8 km
Budapest-Geneva távolság: 989.9 km
Budapest-Antalya távolság: 1518.2 km
Budapest-Amsterdam távolság: 1145.8 km
Budapest-Berlin távolság: 687.5 km
Budapest-Paris távolság: 1246.4 km
Budapest-Rome távolság: 810.0 km
Budapest-Madrid távolság: 1974.1 km
Budapest-Athens távolság: 1125.6 km
Budapest-Vienna távolság: 214.1 km
Budapest-London távolság: 1449.2 km
Budapest-Warsaw távolság: 544.9 km
Budapest-Brussels távolság: 1128.6 km
Budapest-Prague távolság: 444.2 km
Budapest-Milan távolság: 786.8 km
Budapest-Copenhagen távolság: 1013.3 km
Budapest-Dubrovnik távolság: 544.2 km
Budapest-Oslo távolság: 1482.0 km


In [27]:
# Turista sűrűség

# Példa POI-sűrűség számítására
def get_city_crowding_score(lat, lon, radius=10000):
    categories = ["tourism=museum", "tourism=attraction", "amenity=hotel"]
    total_count = 0
    for cat in categories:
        key, value = cat.split("=")
        total_count += query_overpass(lat, lon, key, value, radius)
    return total_count

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row['country'])
    print(f"{city}: {get_city_crowding_score(lat, lon)}")

Barcelona: 230
Lisbon: 131
Tirana: 59
Geneva: 106
Antalya: 32
Amsterdam: 244
Berlin: 362
Paris: 565
Rome: 372
Madrid: 237
Athens: 238
Vienna: 394
London: 449
Warsaw: 526
Brussels: 172
Prague: 385
Milan: 220
Copenhagen: 177
Dubrovnik: 18
Oslo: 103


In [29]:
# Population

from SPARQLWrapper import SPARQLWrapper, JSON

def get_wikidata_population_by_coord(lat, lon, radius_km=10):
    """
    Koordináta alapján lekéri a legközelebbi város népességét Wikidatából.
    radius_km: környező keresési sugár
    """
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    # Koordináta-keresés Wikidata-n (geóval)
    query = f"""
    SELECT ?city ?cityLabel ?population ?coord WHERE {{
      ?city wdt:P31/wdt:P279* wd:Q515;
            wdt:P1082 ?population;
            wdt:P625 ?coord.
      SERVICE wikibase:around {{
        ?city wdt:P625 ?location .
        bd:serviceParam wikibase:center "Point({lon} {lat})"^^geo:wktLiteral .
        bd:serviceParam wikibase:radius "{radius_km}" .
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY DESC(?population)
    LIMIT 1
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    time.sleep(3.5)

    if results["results"]["bindings"]:
        pop = int(results["results"]["bindings"][0]["population"]["value"])
        label = results["results"]["bindings"][0]["cityLabel"]["value"]
        return label, pop
    else:
        return None, None

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row["country"])
    label, pop = get_wikidata_population_by_coord(lat, lon)
    df.loc[city, "population"] = pop
    print(f"{city} -> {label}: {pop}")


Barcelona -> Barcelona: 1702547
Lisbon -> Lisbon: 545796
Tirana -> Tirana: 418495
Geneva -> Geneva: 206635
Antalya -> Antalya: 2426356
Amsterdam -> Amsterdam: 921468
Berlin -> Berlin: 3782202
Paris -> Paris: 2145906
Rome -> Rome: 2748109
Madrid -> Madrid: 3416771
Athens -> Athens: 643452
Vienna -> Vienna: 1973403
London -> London: 8799728
Warsaw -> Warsaw: 1862402
Brussels -> Brussels-Capital Region: 1255795
Prague -> Prague: 1397880
Milan -> Greater Milano: 4934205
Copenhagen -> Copenhagen: 667099
Dubrovnik -> Dubrovnik: 41562
Oslo -> Oslo: 709037


In [31]:
# Értékek normalizálása

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row["country"])
    df.loc[city, "lat"] = lat
    df.loc[city, "lon"] = lon

for city, row in df.iterrows():
    if pd.notna(row.get('population')) and row['population'] > 0:
        # Földrajzi értékek normalizálása
        for geo_col in [col for col in df.columns if col.startswith('geo_')]:
            df.loc[city, f'{geo_col}_per_100k'] = (df.loc[city, geo_col] / row['population']) * 100000
        
        # Életstílus értékek normalizálása
        for lifestyle_col in [col for col in df.columns if col.startswith('eletstilus_')]:
            df.loc[city, f'{lifestyle_col}_per_100k'] = (df.loc[city, lifestyle_col] / row['population']) * 100000
        
        # Turista sűrűség normalizálása
        crowding_score = get_city_crowding_score(row['lat'], row['lon'])  # ehhez előbb menteni kell a koordinátákat
        df.loc[city, 'crowding_per_100k'] = (crowding_score / row['population']) * 100000

C:\Users\Adam\AppData\Local\Temp\ipykernel_24272\386613346.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.loc[city, f'{geo_col}_per_100k'] = (df.loc[city, geo_col] / row['population']) * 100000
C:\Users\Adam\AppData\Local\Temp\ipykernel_24272\386613346.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.loc[city, f'{geo_col}_per_100k'] = (df.loc[city, geo_col] / row['population']) * 100000
C:\Users\Adam\AppData\Local\Temp\ipykernel_24272\386613346.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is u

In [ ]:
# Percentilisek

def calculate_percentiles(df):
    percentiles = {}
    
    # Földrajzi percentilek
    geo_cols = [col for col in df.columns if '_per_100k' in col and 'geo_' in col]
    for col in geo_cols:
        df[f'{col}_percentile'] = df[col].rank(pct=True)
        percentiles[col.replace('geo_', '').replace('_per_100k', '')] = df[f'{col}_percentile'].to_dict()
    
    # Életstílus percentilek
    lifestyle_cols = [col for col in df.columns if '_per_100k' in col and 'eletstilus_' in col]
    for col in lifestyle_cols:
        df[f'{col}_percentile'] = df[col].rank(pct=True)
        percentiles[col.replace('eletstilus_', '').replace('_per_100k', '')] = df[f'{col}_percentile'].to_dict()
    
    # Turista sűrűség percentil
    if 'crowding_per_100k' in df.columns:
        df['crowding_percentile'] = df['crowding_per_100k'].rank(pct=True)
        percentiles['crowding'] = df['crowding_percentile'].to_dict()
    
    return percentiles

# Percentilek számítása
percentile_data = calculate_percentiles(df)

# Cities dictionary frissítése percentilis értékekkel
cities = {}
for city in df.index:
    # Földrajz
    cities[city]["földrajz"] = {
        "tengerpart": percentile_data.get('beach', {}).get(city, 0.5),
        "hegy": percentile_data.get('mountain', {}).get(city, 0.5),
        "város": 0.7,  # ez maradhat fix vagy más metrika alapján
        "sziget": percentile_data.get('island', {}).get(city, 0.5),
        "tópart": percentile_data.get('lake', {}).get(city, 0.5),
        "sivatag": percentile_data.get('desert', {}).get(city, 0.1)
    }
    
    # Életstílus
    cities[city]["életstílus"] = {
        "bulis": percentile_data.get('bulis', {}).get(city, 0.5),
        "relax": percentile_data.get('relax', {}).get(city, 0.5),
        "aktív": 0.5,  # placeholder
        "kulturális": percentile_data.get('kulturalis', {}).get(city, 0.5),
        "családbarát": percentile_data.get('csaladbarat', {}).get(city, 0.5)
    }
    
    # Zsúfoltság
    cities[city]["zsúfoltság"] = percentile_data.get('crowding', {}).get(city, 0.5)

In [ ]:
# Dummy városadatbázis (0-1 normalizált értékek)

# Attribútumok számításának forrása:
## Földrajz:
## Ár: 
## Klíma:
## Életstílus:
## Távolság:
## Zsúfoltság: 

cities = {
    "Lisbon": {
        "földrajz": {"tengerpart": 0.9, "hegy": 0.2, "város": 0.7, "sziget": 0.5, "tópart": 0.2, "sivatag": 0.1},
        "ár": 0.9,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.4, "relax": 1.0, "aktív": 0.5, "kulturális": 1.0, "családbarát": 0.6},
        "távolság": 1.0,
        "zsúfoltság": 0.5,
    },
    "Barcelona": {
        "földrajz": {"tengerpart": 1.0, "hegy": 0.1, "város": 0.9, "sziget": 0.3, "tópart": 0.2, "sivatag": 0.0},
        "ár": 0.5,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.6, "relax": 0.5, "aktív": 0.3, "kulturális": 1.0, "családbarát": 0.5},
        "távolság": 0.5,
        "zsúfoltság": 1.0,
    },
    "Tirana": {
        "földrajz": {"tengerpart": 0.4, "hegy": 0.7, "város": 0.6, "sziget": 0.2, "tópart": 0.3, "sivatag": 0.0},
        "ár": 0.9,
        "klíma": 0.8,
        "életstílus": {"bulis": 0.3, "relax": 0.5, "aktív": 0.6, "kulturális": 1.0, "családbarát": 0.4},
        "távolság": 1.0,
        "zsúfoltság": 0.2,
    }
}

# Dummy user input (csúszkák, multi-select)
user_preferences = {
    "súlyok": {  # nulladik kérdés: mennyire fontos az egyes dimenzió
        "földrajz": 8,
        "ár": 9,
        "klíma": 7,
        "életstílus": 10,
        "távolság": 6,
        "zsúfoltság": 5,
    },
    "földrajz": {"tengerpart": 8, "hegy": 3, "város":5, "sziget":7, "tópart":2, "sivatag":1},
    "ár": 0.9,         
    "klíma": 0.8,      
    "életstílus": {"bulis":4, "relax":8, "aktív":6, "kulturális":7, "családbarát":5},
    "távolság": 0.9,   
    "zsúfoltság": 0.7,
}

In [ ]:
# Normalizált súlyok (összeg=1)
total_weight = sum(user_preferences["súlyok"].values())
weights = {k: v/total_weight for k, v in user_preferences["súlyok"].items()}

# Simple similarity function
def similarity(user_val, city_val):
    return 1 - abs(user_val - city_val)

# Weighted similarity per city
def city_score(user_pref, city_data, weights):
    total = 0
    for attr, weight in weights.items():
        if attr in ["földrajz", "életstílus"]:
            # átlag a multi-select értékekből
            user_vals = user_pref[attr]
            city_vals = city_data[attr]
            sim_vals = []
            for k in user_vals.keys():
                sim_vals.append(similarity(user_vals[k]/10, city_vals[k]))
            avg_sim = sum(sim_vals)/len(sim_vals)
            total += weight * avg_sim
        else:
            # EGYÉNI érték
            user_val = user_pref[attr]
            city_val = city_data[attr]
            sim = similarity(user_val, city_val)
            total += weight * sim
    return total

In [ ]:
# Calculate scores for all cities
city_scores = {}
for city_name, city_data in cities.items():
    score = city_score(user_preferences, city_data, weights)
    city_scores[city_name] = score

# Sort top cities
top_cities = sorted(city_scores.items(), key=lambda x: x[1], reverse=True)

# Output
print("Top ajánlott városok a preferenciáid alapján:")
for city, score in top_cities:
    print(f"{city}: {score:.3f}")